In [ ]:
import scanpy as sc
orig_adata=sc.read_h5ad('hpap_islet_scRNAseq.h5ad')

In [ ]:

alpha_beta_adata=orig_adata[orig_adata.obs['Cell Type'].isin(['Alpha','Beta'])].copy()
# sc.pl.umap(alpha_beta_adata, color=['Cell Type'], frameon=False, legend_loc='on data', title=[],show=False)
AAB_alpha_beta_adata=alpha_beta_adata[alpha_beta_adata.obs['Diabetes Status'].isin(['AAB+','ND'])].copy()
sc.pl.umap(AAB_alpha_beta_adata, color=['Cell Type'], frameon=False, legend_loc='on data', title=[],show=False)

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np

adata=AAB_alpha_beta_adata
cell_type_key = 'cell_type' if 'cell_type' in adata.obs.columns else 'Cell Type'
diabetes_key = 'diabetes_status' if 'diabetes_status' in adata.obs.columns else 'Diabetes Status'

ct_disease_prop = pd.crosstab(
    adata.obs[cell_type_key],
    adata.obs[diabetes_key],
    normalize='index'
)

ct_counts = pd.crosstab(
    adata.obs[cell_type_key],
    adata.obs[diabetes_key]
)

sort_index = ct_counts.sum(axis=1).sort_values(ascending=False).index
ct_disease_prop_sorted = ct_disease_prop.loc[sort_index]
ct_counts_sorted = ct_counts.loc[sort_index]

sns.set_style("whitegrid")
colors = sns.color_palette("Set2", n_colors=ct_counts_sorted.shape[1])
y_positions = np.arange(len(ct_counts_sorted))
bar_height = 0.7
left = np.zeros(len(ct_counts_sorted))

fig, ax = plt.subplots(figsize=(7, 6))

for disease, color in zip(ct_counts_sorted.columns, colors):
    counts = ct_counts_sorted[disease].values
    bars = ax.barh(
        y_positions,
        counts,
        left=left,
        height=bar_height,
        color=color,
        edgecolor="white",
        linewidth=0.5,
        label=disease,
    )
    # for bar, count in zip(bars, counts):
    #     if count == 0:
    #         continue
    #     x = bar.get_x() + bar.get_width() / 2
    #     y = bar.get_y() + bar.get_height() / 2
    #     r, g, b, _ = bar.get_facecolor()
    #     text_color = "black" if 0.299 * r + 0.587 * g + 0.114 * b > 0.6 else "white"
    #     ax.text(x, y, f"{int(count)}", ha="center", va="center", fontsize=8, color=text_color)
    left += counts

ax.set_yticks(y_positions)
ax.set_yticklabels(ct_counts_sorted.index)
max_total = ct_counts_sorted.sum(axis=1).max()
ax.set_xlim(0, max_total * 1.15)
ax.set_xlabel("Cell count")
ax.set_ylabel("Cell type")
ax.set_title("Cell type counts by diabetes status (sorted by total cells)")
ax.legend(title="Disease", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="x", linestyle="--", alpha=0.4)

for y_pos, total in enumerate(ct_counts_sorted.sum(axis=1).values):
    ax.text(total * 1.02, y_pos, f"Total: {int(total)}", ha="left", va="center", fontsize=9, fontweight="bold")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig("cell_type_disease_counts.pdf", bbox_inches="tight")
plt.show()


In [ ]:
adata.obs['cell_type']=adata.obs['Cell Type']
del adata.obs['Cell Type']
adata.obs['diabetes_status']=adata.obs['Diabetes Status']
del adata.obs['Diabetes Status']

In [ ]:
import numpy as np
# Convert log-normalized data back to raw counts and store in count layer
adata.layers['count'] = np.expm1(adata.X)

In [ ]:
beta_adata = adata[adata.obs['cell_type']=='Beta']
beta_adata.obs['diabetes_status'].value_counts()

In [ ]:
alpha_adata = adata[adata.obs['cell_type']=='Alpha']
alpha_adata.obs['diabetes_status'].value_counts()

In [ ]:

adata.write("processed_hpap.h5ad")

In [ ]:
import scanpy as sc
adata=sc.read_h5ad("processed_hpap.h5ad")
adata

In [ ]:
subset=adata[adata.obs['cell_type'].isin(['Beta','Alpha'])]
subset=subset[subset.obs['Sex']=='M']
subset=subset[subset.obs['diabetes_status'].isin(['AAB+','ND'])]
subset.write("processed_hpap_male.h5ad")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42
# Count the occurrences of each diabetes status
beta_subset=subset[subset.obs['cell_type']=='Beta']
status_counts = beta_subset.obs['diabetes_status'].value_counts()

# Create a bar plot
plt.figure(figsize=(6, 4))
sns.barplot(x=status_counts.index, y=status_counts.values, palette='pastel')
plt.title('Count of Diabetes Status in Beta Cells')
plt.xlabel('Diabetes Status',weight='bold', size=18)
plt.ylabel('Count',weight='bold', size=18)
plt.yticks([0,max(status_counts.values)//2,max(status_counts.values)],size=14)
plt.xticks(size=14)
sns.despine()
plt.savefig("beta_cell_diabetes_status_counts.pdf", bbox_inches="tight")
plt.show()